# FinTailor - Qwen3.5 LoRA fine-tune (Colab)
Runtime -> Change runtime type -> **T4 GPU** (free) / L4 or A100 (Pro).

Before running: on the PC run `python scripts/make_colab_bundle.py`, then upload the resulting
`fintailor_colab.zip` to Google Drive at **`MyDrive/fintailor/fintailor_colab.zip`** (one file; no API keys or DB inside).
The next cell unzips it. Training outputs (checkpoints, adapter, GGUF) are saved to `MyDrive/fintailor/lora_4b`.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, zipfile
BUNDLE = '/content/drive/MyDrive/fintailor/fintailor_colab.zip'
assert os.path.exists(BUNDLE), 'Upload fintailor_colab.zip to MyDrive/fintailor/ first'
with zipfile.ZipFile(BUNDLE) as z:
    z.extractall('/content/fintailor')          # local disk: faster than reading from Drive
REPO = '/content/fintailor/repo'
DATA = '/content/fintailor/data'
OUT = '/content/drive/MyDrive/fintailor/lora_4b'  # on Drive so it survives disconnects
os.makedirs(OUT, exist_ok=True)
%cd $REPO
!ls $DATA

In [ ]:
%%capture
!pip install -q unsloth
!pip install -q python-dotenv pydantic duckdb

In [ ]:
# Optional: Hugging Face upload / W&B. Not needed - outputs are saved to Google Drive.
# from huggingface_hub import login
# login()  # paste a write token
# import wandb; wandb.login()

In [ ]:
# Mix: 7000 news / 2000 explain / 1000 general finance Q&A (FinGPT sentiment excluded: fake event labels)
!python training/mix_datasets.py --data $DATA --out /content/train_mix.jsonl

In [ ]:
# 1) 100-step timing test (T4 kernel compile is slow - measure first)
!python training/finetune_lora.py --data /content/train_mix.jsonl --model unsloth/Qwen3.5-2B --out /content/test_run --max-steps 100

In [ ]:
# 2) Stage 1: Qwen3.5-4B, 16-bit LoRA r16/a16, seq 2048, 5-10k examples, 1 epoch
!python training/finetune_lora.py --data /content/train_mix.jsonl --eval-data $DATA/news_test.jsonl --model unsloth/Qwen3.5-4B --out $OUT --max-examples 10000 --export # --hub-repo YOURNAME/fintailor-qwen35-4b-lora

In [ ]:
# Resume after a disconnect (checkpoints every 100 steps on Drive)
# !python training/finetune_lora.py --data /content/train_mix.jsonl --model unsloth/Qwen3.5-4B --out $OUT --resume

In [ ]:
# Quick sanity check of the adapter
from unsloth import FastModel
import json, sys
sys.path.insert(0, REPO)
from fintailor.llm.prompts import NEWS_SYSTEM
model, tok = FastModel.from_pretrained(OUT + '/final_adapter', max_seq_length=2048, load_in_4bit=False)
t = getattr(tok, 'tokenizer', tok)
msgs = [{'role':'system','content':NEWS_SYSTEM},{'role':'user','content':'Headlines:\n1. [XOM] Oil jumps 6% after attack on Gulf tankers'}]
ids = t.apply_chat_template(msgs, add_generation_prompt=True, enable_thinking=False, return_tensors='pt').to('cuda')
print(t.decode(model.generate(input_ids=ids, max_new_tokens=300)[0][ids.shape[1]:]))

## Stage 2 (Colab Pro, after Gate 1)
`--model unsloth/Qwen3.5-9B --rank 32 --alpha 32 --seq 4096 --lr 1e-4 --epochs 2 --max-examples 30000`

## Serve locally
Download `gguf/*.gguf` and run `llama-server -m model-q4_k_m.gguf --port 8080`, then set `LLM_BASE_URL=http://localhost:8080/v1` and `LLM_MODEL=fintailor` in `.env`.
Gate 1: `python training/eval_llm.py --news .../news_test.jsonl --base http://localhost:8081/v1:base --tuned http://localhost:8080/v1:fintailor`